# TNFR Spectral Factorization Regression & Telemetry Notebook


This notebook accompanies the spectral factorization lab. It exercises both regression paths (FFT-enabled and arithmetic-cache) and captures `SpectralAnalysisResult` telemetry for exploratory plotting.

## 1. Configure Environment and Imports


Prepare deterministic settings, import scientific libraries, and wire the spectral factorizer module for notebook execution.

In [2]:
%matplotlib inline

import math
import os
import sys
import time
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "factorization-lab").exists():
    for parent in REPO_ROOT.parents:
        candidate = parent / "factorization-lab"
        if candidate.exists():
            REPO_ROOT = parent
            break

LAB_ROOT = REPO_ROOT / "factorization-lab"
if LAB_ROOT.exists() and str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from tnfr_factorization import SpectralPaleyFactorizer  # type: ignore[import]
import tnfr_factorization.spectral_paley as sp  # type: ignore[import]

sns.set_theme(style="whitegrid")
np.random.seed(42)


## 2. Load or Stub Spectral Analysis Components


Instantiate canonical analyzers and lightweight stubs so we can deterministically trigger FFT-enabled and arithmetic-cache behaviors.

In [3]:
class DeterministicSpectralState:
    """Minimal container mimicking TNFR spectral state outputs."""

    def __init__(self, node_count: int, coherence_length: float = 4.0) -> None:
        self.eigenvalues = np.linspace(0.0, 1.0, max(node_count, 1), dtype=float)
        self.coherence_length = coherence_length


class RecordingFFTEngine:
    """Notebook-friendly stub that records how often it is used."""

    def __init__(self, coherence_length: float = 4.0) -> None:
        self.coherence_length = coherence_length
        self.calls: List[int] = []

    def get_spectral_state(self, graph: Any, force_recompute: bool = False) -> DeterministicSpectralState:
        self.calls.append(graph.number_of_nodes())
        return DeterministicSpectralState(graph.number_of_nodes(), self.coherence_length)


def build_factorizers() -> Dict[str, SpectralPaleyFactorizer]:
    """Return both canonical and stubbed factorizers for experiments."""

    canonical = SpectralPaleyFactorizer()
    stubbed = SpectralPaleyFactorizer(fft_engine=RecordingFFTEngine(coherence_length=6.25))
    return {"canonical": canonical, "stubbed": stubbed}


factories = build_factorizers()
factories

{'canonical': <tnfr_factorization.spectral_paley.SpectralPaleyFactorizer at 0x223aee792b0>,
 'stubbed': <tnfr_factorization.spectral_paley.SpectralPaleyFactorizer at 0x223aeeada90>}

## 3. Add Regression Tests for FFT-Enabled Path


Validate that the FFT-enabled branch produces deterministic Laplacian gaps and that the stubbed engine records spectral calls.

In [5]:
def test_fft_branch_laplacian_gap_consistency(n: int = 37) -> None:
    factorizer = factories["stubbed"]
    engine = factorizer._fft_engine  # type: ignore[attr-defined]

    result = factorizer.analyze(n)

    expected_eigs = np.linspace(0.0, 1.0, result.node_count, dtype=float)
    expected_gap = next((val for val in expected_eigs if val > 1e-9), 0.0)

    diff = abs(result.laplacian_gap - expected_gap)
    assert diff <= 1e-9
    assert np.isclose(result.coherence_length, engine.coherence_length)
    assert len(engine.calls) >= 1

    print(
        f"PASS: gap={result.laplacian_gap:.6f}, phase_gradient={result.phase_gradient:.6f}, calls={len(engine.calls)}"
    )


test_fft_branch_laplacian_gap_consistency()


PASS: gap=0.027778, phase_gradient=0.000751, calls=2


## 4. Add Regression Tests for Arithmetic Cache Path


Exercise the canonical arithmetic telemetry cache to ensure ΔNFR data is memoized across repeated analyses.

In [6]:
def test_arithmetic_cache_hits(n: int = 221) -> None:
    factorizer = factories["canonical"]

    cache_clear = getattr(sp._compute_arithmetic_telemetry, "cache_clear", None)
    if callable(cache_clear):
        cache_clear()

    original_prime_factorization = sp._prime_factorization
    call_count = {"value": 0}

    def _instrumented_prime_factorization(value: int) -> Dict[int, int]:
        call_count["value"] += 1
        return original_prime_factorization(value)

    sp._prime_factorization = _instrumented_prime_factorization  # type: ignore[assignment]
    try:
        first = factorizer.analyze(n)
        second = factorizer.analyze(n)
    finally:
        sp._prime_factorization = original_prime_factorization  # type: ignore[assignment]

    assert call_count["value"] == 1, "prime factorization should be memoized"
    assert np.isclose(first.arithmetic_delta_nfr, second.arithmetic_delta_nfr)
    assert first.candidate_factors == second.candidate_factors

    print(
        f"PASS: cache hits verified for n={n}, factors={first.candidate_factors}, ΔNFR={first.arithmetic_delta_nfr:.3e}"
    )


test_arithmetic_cache_hits()

PASS: cache hits verified for n=221, factors=[], ΔNFR=3.259e+00


## 5. Capture SpectralAnalysisResult History


Sweep both factorizers over a deterministic set of integers, persist the telemetry into a pandas DataFrame, and retain timestamps plus energy-derived metrics for downstream plotting.

In [7]:
def collect_history(samples: List[int] | None = None) -> pd.DataFrame:
    samples = samples or [185, 221, 289, 299, 323]
    records: List[Dict[str, Any]] = []

    for route, factorizer in factories.items():
        for n in samples:
            start = time.perf_counter()
            result = factorizer.analyze(n)
            elapsed_ms = (time.perf_counter() - start) * 1000.0
            records.append(
                {
                    "route": route,
                    "n": n,
                    "laplacian_gap": result.laplacian_gap,
                    "phase_gradient": result.phase_gradient,
                    "coherence_score": result.coherence_score,
                    "phi_s": result.phi_s,
                    "delta_nfr": result.arithmetic_delta_nfr,
                    "timestamp": time.time(),
                    "elapsed_ms": elapsed_ms,
                    "candidate_factors": ",".join(map(str, result.candidate_factors)) or "-",
                }
            )
    frame = pd.DataFrame.from_records(records).sort_values(["route", "n"]).reset_index(drop=True)
    return frame


history_df = collect_history()
history_df

,route,n,laplacian_gap,phase_gradient,coherence_score,phi_s,delta_nfr,timestamp,elapsed_ms,candidate_factors
0,canonical,185,64.699265,0.349726,0.299905,0.391304,3.315722,1.764519e+09,31.6738,5
1,canonical,221,88.066966,0.398493,0.253543,0.436364,3.259307,1.764519e+09,13.1788,-
2,canonical,289,272.000000,0.941176,0.031244,0.944444,2.091038,1.764519e+09,82.3741,-
3,canonical,299,116.825324,0.388124,0.262760,0.420000,3.249823,1.764519e+09,70.1315,-
4,canonical,323,93.944487,0.289060,0.367879,0.370370,3.244294,1.764519e+09,53.7964,-
5,stubbed,185,0.005435,0.000029,0.431711,0.391304,3.315722,1.764519e+09,4.8280,-
6,stubbed,221,0.004545,0.000021,0.431711,0.436364,3.259307,1.764519e+09,7.6035,-
7,stubbed,289,0.003472,0.000012,0.344742,0.944444,2.091038,1.764519e+09,23.4383,-
8,stubbed,299,0.003333,0.000011,0.431711,0.420000,3.249823,1.764519e+09,14.4836,-
9,stubbed,323,0.003086,0.000009,0.431711,0.370370,3.244294,1.764519e+09,15.0211,-


## 6. Plot SpectralAnalysisResult History


Use seaborn/matplotlib visualizations plus a lightweight widget to compare FFT vs arithmetic-cache runs and highlight any regression anomalies.

In [8]:
from IPython.display import display
import ipywidgets as widgets

METRIC_OPTIONS = [
    "laplacian_gap",
    "phase_gradient",
    "coherence_score",
    "phi_s",
    "delta_nfr",
]


def plot_metric(metric: str) -> None:
    pivot = history_df.pivot(index="n", columns="route", values=metric)
    plt.figure(figsize=(8, 4))
    for route in pivot.columns:
        plt.plot(pivot.index, pivot[route], marker="o", label=route.title())
    plt.title(f"{metric} vs n")
    plt.xlabel("n")
    plt.ylabel(metric)
    plt.legend()
    plt.show()

    plt.figure(figsize=(6, 4))
    sns.heatmap(pivot.fillna(0.0), annot=True, fmt=".3f", cmap="viridis")
    plt.title(f"Heatmap of {metric} (route × n)")
    plt.show()


metric_selector = widgets.Dropdown(options=METRIC_OPTIONS, value="laplacian_gap", description="Metric")
widgets.interact(plot_metric, metric=metric_selector)

display(history_df)

interactive(children=(Dropdown(description='Metric', options=('laplacian_gap', 'phase_gradient', 'coherence_sc…

,route,n,laplacian_gap,phase_gradient,coherence_score,phi_s,delta_nfr,timestamp,elapsed_ms,candidate_factors
0,canonical,185,64.699265,0.349726,0.299905,0.391304,3.315722,1.764519e+09,31.6738,5
1,canonical,221,88.066966,0.398493,0.253543,0.436364,3.259307,1.764519e+09,13.1788,-
2,canonical,289,272.000000,0.941176,0.031244,0.944444,2.091038,1.764519e+09,82.3741,-
3,canonical,299,116.825324,0.388124,0.262760,0.420000,3.249823,1.764519e+09,70.1315,-
4,canonical,323,93.944487,0.289060,0.367879,0.370370,3.244294,1.764519e+09,53.7964,-
5,stubbed,185,0.005435,0.000029,0.431711,0.391304,3.315722,1.764519e+09,4.8280,-
6,stubbed,221,0.004545,0.000021,0.431711,0.436364,3.259307,1.764519e+09,7.6035,-
7,stubbed,289,0.003472,0.000012,0.344742,0.944444,2.091038,1.764519e+09,23.4383,-
8,stubbed,299,0.003333,0.000011,0.431711,0.420000,3.249823,1.764519e+09,14.4836,-
9,stubbed,323,0.003086,0.000009,0.431711,0.370370,3.244294,1.764519e+09,15.0211,-


## 7. Load Benchmark Snapshot

Reference the automated Paley gap benchmark output to keep notebook analyses aligned with the reproducibility artifacts recorded under `results/benchmarks/`.

In [ ]:
import json
from pathlib import Path

benchmark_dir = (Path(__file__).resolve().parent / "../results/benchmarks").resolve()
smoke_path = benchmark_dir / "paley_gap_smoke.json"
extended_path = benchmark_dir / "paley_gap_extended.json"


def _load(path: Path) -> tuple[pd.DataFrame, float]:
    payload = json.loads(path.read_text())
    frame = pd.DataFrame(payload["records"])
    frame["benchmark"] = path.name
    return frame, payload.get("timestamp", 0.0)


smoke_df, smoke_ts = _load(smoke_path)
extended_df, extended_ts = _load(extended_path)
combined_df = pd.concat([smoke_df, extended_df], ignore_index=True)

print(f"Loaded smoke benchmark: {smoke_path} (ts={smoke_ts})")
display(smoke_df)
print(f"Loaded extended benchmark: {extended_path} (ts={extended_ts})")
display(extended_df)
combined_df